### set up of the Jupyter Notebook
set up is done by the developer
of the underling python modules

In [1]:
from mstr_robotics._paths import REPO_ROOT, USER_CONFIG, OSI_FILES, OSI_SCHEMA, OSI_DASHBOARD_CONTEXT, MCP_DATA, PYTHON_IO
import pandas as pd
import random
import uuid
import json
from mstr_robotics import regam
from mstr_robotics._helper import Misc
from mstr_robotics.mstr_classes import get_conn,MdSearches
from mstr_robotics.report import Prompts

run_id = uuid.uuid1().__str__()
i_test=regam.TestExe()
#i_parse_pa = ParsePa()
#i_dossiers=dossiers()
i_msic=Misc()
i_prompts=Prompts()
i_get_conn=get_conn
i_regam=regam.Regam({"run_id": run_id})
i_md_search=MdSearches()
i_regam_jobs=regam.RegamJobs()

with open(USER_CONFIG, 'r') as openfile:
    user_d = json.load(openfile)

#set user credentials and open a connection to the i-server

project_id="B7CA92F04B9FAE8D941C3E9B7E0CD754"
pa_project_id="7576CD5F48607C21C914ACBE053B259B"
REGAM_cube_folder_id="C57AFAEC4DE90EE2DE84BC8155D32DE8"
hier_att_cube_id="DC78107F4DE830DF127FA592C1EB7E68"

pa_base_url= user_d["conn_params"]["base_url"]

conn_params =  user_d["conn_params"]
conn_params["project_id"]=project_id
#conn = Connection(**conn_params)
conn=get_conn(**conn_params)
conn.select_project(project_id)
conn.headers['Content-type'] = "application/json"
           
pa_conn=get_conn(**conn_params)
pa_conn.select_project(pa_project_id)

def select_rows_by_job_id(pa_raw_data_df, job_id=None):
    """
    Select all rows belonging to a specific Job@ID
    If job_id is None, randomly selects the first available Job@ID
    """
    if job_id is None:
        unique_job_ids = pa_raw_data_df["Job@ID"].unique()
        job_id = unique_job_ids[0] if len(unique_job_ids) > 0 else None
    
    if job_id is None:
        return pd.DataFrame()
    
    filtered_df = pa_raw_data_df[pa_raw_data_df["Job@ID"] == job_id]
    return filtered_df

hier_att_df=i_regam.refresh_hier_att_cube(conn=conn,REGAM_cube_folder_id=REGAM_cube_folder_id,proj_prp_hier_l=None,
                      hier_att_cube_id=hier_att_cube_id)

hier_att_df=i_regam.load_hier_att_df(conn=conn,hier_att_cube_id=hier_att_cube_id)


Connection to Strategy One Intelligence Server has been established.
Connection to Strategy One Intelligence Server has been established.
SuperCube object named: 'System hier_att' with ID: 'DC78107F4DE830DF127FA592C1EB7E68'


Uploading 1/1:   0%|                                                  | 0/1 [00:00<?, ?it/s]

Error uploading data to dataset DC78107F4DE830DF127FA592C1EB7E68
Could not decode the response from the I-Server. Please check if I-Server is running correctly


Uploading 1/1: 100%|████████████████████████████████| 1/1 [00:00<00:00, 11.72it/s, rows=215]

Error publishing uploaded data for dataset with ID DC78107F4DE830DF127FA592C1EB7E68 Cancelling publication.
Could not decode the response from the I-Server. Please check if I-Server is running correctly


Super cube 'System hier_att' published successfully.
_Cube object named: 'System hier_att' with ID: 'DC78107F4DE830DF127FA592C1EB7E68'


In [2]:
#get the jobs stored in a static report for this use case.
pa_report_id="31FF880045D230BE04C209A42C108B24"
pa_conn.select_project(pa_project_id)
pa_raw_data_df=i_regam.fetch_pa_rep_jobs(pa_conn=pa_conn, pa_report_id=pa_report_id)
#identify projects and prompts
#pa_prp_id_l = i_parse_pa.get_pa_prp_id_l(get_pa_raw_data_df=pa_raw_data_df)
selected_rows = select_rows_by_job_id(pa_raw_data_df)
selected_rows.head(5)

,Date@ID,Session@ID,Job@ID,Project@Name,Project@GUID,Object@Name,Object@GUID,Prompt_Type@DESC,Prompt_Type@ID,Prompt@Name,Prompt@GUID,Prompt_Answer@ID,Prompt_Answer_Sequence@ID,Sum Action CPU Duration (s)
0,1/16/2026,7417833397931741184,4333,MicroStrategy Tutorial,B7CA92F04B9FAE8D941C3E9B7E0CD754,Customer Detail report (Dashboard),C060EF484E1352B26D945CBAF191F2B8,Attribute Element Prompt,2,Choose from the Elements of Customer,31A96A44479DBB67AA74B381CE8348DE,,1,4.75
1,1/16/2026,7417833397931741184,4333,MicroStrategy Tutorial,B7CA92F04B9FAE8D941C3E9B7E0CD754,Customer Detail report (Dashboard),C060EF484E1352B26D945CBAF191F2B8,Attribute Element Prompt,2,Elements of Call Center,76A492374445A79DE945CD991068419C,,1,3.50
2,1/16/2026,7417833397931741184,4333,MicroStrategy Tutorial,B7CA92F04B9FAE8D941C3E9B7E0CD754,Customer Detail report (Dashboard),C060EF484E1352B26D945CBAF191F2B8,Attribute Element Prompt,2,Elements of Customer Region,BE16BFE74B63506CA59FE3AA71174916,,1,3.50
3,1/16/2026,7417833397931741184,4333,MicroStrategy Tutorial,B7CA92F04B9FAE8D941C3E9B7E0CD754,Customer Detail report (Dashboard),C060EF484E1352B26D945CBAF191F2B8,Attribute Element Prompt,2,Elements of Employee,EBCB12A0469E6FA5D49800B25B33E0D3,,1,3.50
4,1/16/2026,7417833397931741184,4333,MicroStrategy Tutorial,B7CA92F04B9FAE8D941C3E9B7E0CD754,Customer Detail report (Dashboard),C060EF484E1352B26D945CBAF191F2B8,Attribute Element Prompt,2,Elements of Manager,7DA2E6DD4BF14BAFCCFDB7B5A38BE70E,,1,3.50


In [3]:
all_jobs_prp_ans_JSON_d=i_regam.run_bld_job_prp_JSON(conn,pa_raw_data_df=selected_rows,hier_att_df=hier_att_df)
all_jobs_prp_ans_JSON_d

[]
[{'id': '31A96A44479DBB67AA74B381CE8348DE', 'type': 'ELEMENTS', 'answers': []}, {'id': '76A492374445A79DE945CD991068419C', 'type': 'ELEMENTS', 'answers': []}, {'id': 'EBCB12A0469E6FA5D49800B25B33E0D3', 'type': 'ELEMENTS', 'answers': []}, {'id': '7DA2E6DD4BF14BAFCCFDB7B5A38BE70E', 'type': 'ELEMENTS', 'answers': []}, {'id': '7E31364947BC9CB5B49EB39367F0DC1F', 'type': 'ELEMENTS', 'answers': [{'id': 'h2021;8D679D5111D3E4981000E787EC6DE8A4', 'name': '2021'}]}, {'id': 'BE16BFE74B63506CA59FE3AA71174916', 'type': 'ELEMENTS', 'answers': []}]
[]
[{'id': '31A96A44479DBB67AA74B381CE8348DE', 'type': 'ELEMENTS', 'answers': []}, {'id': '76A492374445A79DE945CD991068419C', 'type': 'ELEMENTS', 'answers': []}, {'id': 'EBCB12A0469E6FA5D49800B25B33E0D3', 'type': 'ELEMENTS', 'answers': []}, {'id': '7DA2E6DD4BF14BAFCCFDB7B5A38BE70E', 'type': 'ELEMENTS', 'answers': []}, {'id': '7E31364947BC9CB5B49EB39367F0DC1F', 'type': 'ELEMENTS', 'answers': [{'id': 'h2021;8D679D5111D3E4981000E787EC6DE8A4', 'name': '2021'

{'run_id': 'b344c9a1-6d46-11f1-a465-2c9c5850f942',
 'all_jobs_prp_ans_JSON_l': [{'run_id': 'b344c9a1-6d46-11f1-a465-2c9c5850f942',
   'proj_id': 'B7CA92F04B9FAE8D941C3E9B7E0CD754',
   'report_id': 'C060EF484E1352B26D945CBAF191F2B8',
   'report_name': 'Customer Detail report (Dashboard)',
   'session': '7417833397931741184',
   'rep_job': '4333',
   'sucsess_fg': True,
   'cnt_prompts': 6,
   'prompt_ans': '{"prompts": [{"id": "31A96A44479DBB67AA74B381CE8348DE", "type": "ELEMENTS", "answers": []}, {"id": "76A492374445A79DE945CD991068419C", "type": "ELEMENTS", "answers": []}, {"id": "EBCB12A0469E6FA5D49800B25B33E0D3", "type": "ELEMENTS", "answers": []}, {"id": "7DA2E6DD4BF14BAFCCFDB7B5A38BE70E", "type": "ELEMENTS", "answers": []}, {"id": "7E31364947BC9CB5B49EB39367F0DC1F", "type": "ELEMENTS", "answers": [{"id": "h2021;8D679D5111D3E4981000E787EC6DE8A4", "name": "2021"}]}, {"id": "BE16BFE74B63506CA59FE3AA71174916", "type": "ELEMENTS", "answers": []}]}'},
  {'run_id': 'b344c9a1-6d46-11f1-a4

In [4]:
i_test.run_test_exe(conn=conn,all_jobs_prp_ans_JSON_l=all_jobs_prp_ans_JSON_d["all_jobs_prp_ans_JSON_l"])

Report: Customer Detail report (Dashboard)_C060EF484E1352B26D945CBAF191F2B8_sess_7417833397931741184_rep_job4333 has been created successfully
Report: Customer Detail report (Dashboard)_C060EF484E1352B26D945CBAF191F2B8_sess_7417833397931741184_rep_job4333 has been created successfully
Report: Customer Detail report (Dashboard)_C060EF484E1352B26D945CBAF191F2B8_sess_7417833397931741184_rep_job4333 has been created successfully
Report: Customer Detail report (Dashboard)_C060EF484E1352B26D945CBAF191F2B8_sess_7417833397931741184_rep_job4333 has been created successfully
Report: Customer Detail report (Dashboard)_C060EF484E1352B26D945CBAF191F2B8_sess_7417833397931741184_rep_job4333 has been created successfully
Report: Customer Detail report (Dashboard)_C060EF484E1352B26D945CBAF191F2B8_sess_7417833397931741184_rep_job4333 has been created successfully
